# EGS 2405 Geostatistics — Week 0: the ideas you need before Week 1

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanibenson/jkuat-geospatial-teaching/blob/main/egs2405/week0-prerequisites.ipynb)

**Self-study, ungraded, about three hours.** Do it before the first lecture.

The reader assumes you are comfortable with some statistics and a little linear
algebra that most of you met in second year and have since half-forgotten. This
notebook brings those back, one idea at a time. Each section is built the same way:

1. **The idea**, in plain language.
2. **A small demonstration** on made-up data where we know the right answer.
3. **Check yourself**: a short exercise. Run the check cell underneath and it tells
   you ✓, ✗ with a hint, or ○ not attempted yet. A model answer is folded away
   below each exercise. Try first, then look.
4. **Where this goes**: the chapter of the reader that leans on it.

This notebook is about **concepts**, not Python syntax. If the code itself is the
obstacle, do the *Python refresher and self-check* appendix of the reader first.

### Skip what you already know

Answer these in your head. If you are confident about a question, you can skim that
section. If you hesitate, do the section properly.

1. Why are 200 soil samples from one farm not "200 independent observations"?
2. For right-skewed data, which is bigger, the mean or the median? What does
   $\exp(\text{mean of }\log x)$ estimate?
3. Complete this: $\operatorname{Var}(X-Y) = \dots$
4. Why is a distance of "0.05" between two GPS points meaningless for analysis?
5. What does `scipy.optimize.curve_fit` minimise, and why does it need a starting guess?
6. Two rain gauges, one twice as precise as the other: what weights give the best average?
7. A prediction is 20 with standard deviation 4. What is the probability that the true value is below 15?
8. Why can random cross-validation flatter a spatial model?

<details><summary>Short answers</summary>

1. Nearby samples resemble each other, so each new one adds less new information
   than an independent draw would. (§1)
2. The mean. $\exp(\overline{\log x})$ estimates the *median*, not the mean. (§2)
3. $\operatorname{Var}X + \operatorname{Var}Y - 2\operatorname{Cov}(X,Y)$. This single line is
   where the variogram comes from. (§3)
4. It is in degrees. A degree is not a fixed length and is not a unit you can
   report a range in. Project to metres first. (§4)
5. The sum of squared residuals. It searches iteratively and can land on the wrong
   minimum or fail to converge if it starts far from the answer. (§5)
6. Weights proportional to $1/\sigma^2$: 0.8 and 0.2. (§6)
7. $\Phi\big((15-20)/4\big) = \Phi(-1.25) \approx 0.106$. (§7)
8. Held-out points have close neighbours left in the training set, so the test is
   much easier than predicting into a real gap. (§8)
</details>

In [ ]:
# Setup: run this cell first, every time you open the notebook.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize
from scipy.spatial.distance import cdist

plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})


def check(label, value, test, hint=""):
    # ✓ / ✗ / ○ feedback for the exercises. It never raises, so 'Run all' is safe.
    if value is ...:
        print(f"○ {label}: not attempted yet")
        return
    try:
        ok = bool(test(value))
    except Exception as e:
        ok, hint = False, f"{hint}  (checking raised {type(e).__name__}: {e})"
    print(("✓ " if ok else "✗ ") + label + ("" if ok else f"\n   Hint: {hint}"))


def smooth_transect(n, corr_len, rng):
    # A 1-D random field of length n: white noise blurred by a Gaussian kernel,
    # scaled so that every value has mean 0 and variance 1. Larger corr_len
    # makes neighbouring values more alike.
    half = int(4 * corr_len)
    k = np.exp(-0.5 * (np.arange(-half, half + 1) / corr_len) ** 2)
    k /= np.sqrt((k ** 2).sum())
    return np.convolve(rng.standard_normal(n + 2 * half), k, mode="valid")

print("Ready.")

---
## 1 · Random variables, and why one field is only one draw

### The idea

A **random variable** is a quantity whose value we cannot know in advance but whose
*behaviour* we can describe: its average (the **expectation**, $E[X]$) and its spread
(the **variance**, $\operatorname{Var}X$, whose square root is the **standard deviation** $\sigma$).

Classical statistics runs on one assumption above all: **independent** repeated draws.
Take $n$ independent samples and the sample mean wobbles around the true mean with
standard deviation $\sigma/\sqrt{n}$. More data means a smaller wobble. You know this rule.

Soil carbon over a farm does not behave like that. Two cores taken a metre apart are
nearly the same. Two cores a kilometre apart are not. The values are **dependent**, and
that changes everything:

* Two hundred samples from one farm carry far less information than two hundred
  independent draws. The $\sigma/\sqrt{n}$ rule is badly optimistic.
* We only ever see **one** field. Geostatistics treats that field as one draw (one
  *realisation*) from a random process, and has to learn about the process from
  that single draw. This is the one-realisation problem, and it is the first idea
  of Week 1.

### Demonstration

In [ ]:
rng = np.random.default_rng(1)

# Part A: independent draws. The sampling distribution of the mean behaves as promised.
n, sigma = 200, 1.0
means_indep = rng.normal(0, sigma, size=(2000, n)).mean(axis=1)
print(f"independent: sd of 2000 sample means = {means_indep.std():.3f}; "
      f"theory sigma/sqrt(n) = {sigma/np.sqrt(n):.3f}")

# Part B: two realisations of the SAME spatially correlated process.
fig, ax = plt.subplots()
for seed in (2, 3):
    z = smooth_transect(n, corr_len=10, rng=np.random.default_rng(seed))
    ax.plot(z, lw=1.2, label=f"realisation (seed {seed}): mean = {z.mean():+.2f}")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="position along transect (m)", ylabel="value",
       title="Same process, same true mean (0), two very different transects")
ax.legend(); plt.show()

Both transects come from a process whose true mean is exactly 0 and whose variance is
exactly 1. Yet each has long runs above or below zero, and its own sample mean can sit
far from 0, much further than $1/\sqrt{200} \approx 0.07$ would allow.

### Check yourself

Simulate 2000 transects of length 200 with `corr_len=10` and work out how much their
means actually vary. Then turn that into an **effective sample size**:

$$ n_\text{eff} = \frac{\sigma^2}{\operatorname{Var}(\bar z)} $$

This is the number of *independent* samples that would give a mean this precise.

In [ ]:
rng = np.random.default_rng(10)
# 1. Build a (2000, 200) array: 2000 transects from smooth_transect(200, 10, rng)
# 2. sd_means = standard deviation of the 2000 transect means
# 3. n_eff = 1 / sd_means**2      (sigma is 1 here)
sd_means = ...
n_eff = ...

<details><summary>Show a model answer</summary>

```python
rng = np.random.default_rng(10)
Z = np.array([smooth_transect(200, 10, rng) for _ in range(2000)])
sd_means = Z.mean(axis=1).std()
n_eff = 1 / sd_means**2
print(f"sd of means = {sd_means:.3f}  (independent theory: {1/np.sqrt(200):.3f})")
print(f"n_eff = {n_eff:.1f} out of 200 samples")
```

</details>

In [ ]:
check("1a spread of the transect means", sd_means,
      lambda v: 0.3 < v < 0.55,
      "Take the mean of each transect (axis=1), then the std of those 2000 numbers.")
check("1b effective sample size", n_eff,
      lambda v: 3 < v < 12,
      "n_eff = 1 / sd_means**2. It should be a small number, far below 200.")

Two hundred samples, worth about six independent ones. Hold on to that number. It
explains why the reader keeps insisting on *honest* error estimates.

> **Where this goes:** Week 1, *The random function, and the one-realisation problem*
> and *Three grades of stationarity*. Stationarity is the assumption that lets us pool
> pairs from different parts of the one field as if they were repeated draws.

---
## 2 · Distributions, skew and transforms

### The idea

Environmental concentrations (soil carbon, heavy metals, rainfall totals) are
rarely bell-shaped. They are **right-skewed**: most values are modest, and a long tail
of high values pulls the **mean** above the **median**. Several tools you will use:

* **Quantiles.** The $p$-quantile is the value below which a fraction $p$ of the data
  falls. The median is the 0.5-quantile. The **empirical CDF** plots, for every value,
  the fraction of data at or below it. Indicator geostatistics (Week 9) is built
  entirely on these.
* **The log transform** often makes skewed data roughly normal. The trap: the
  back-transform of the mean of the logs, $\exp(\overline{\log x})$, is **not** the mean.
  It estimates the median. Many published maps are too low for this reason.
* **The Q–Q plot** checks normality. Data quantiles are plotted against normal
  quantiles, and a straight line means "normal enough".
* **The normal-score transform** is a stronger tool. It replaces each value by the
  normal quantile of its rank. The result is exactly normal by construction, and
  Sequential Gaussian Simulation (Week 8) requires it.

### Demonstration

In [ ]:
rng = np.random.default_rng(20)
soc = rng.lognormal(mean=np.log(18), sigma=0.5, size=300)   # 'soil organic carbon', g/kg

print(f"mean {soc.mean():.1f}   median {np.median(soc):.1f}   "
      f"exp(mean(log)) {np.exp(np.log(soc).mean()):.1f}   skewness {stats.skew(soc):.2f}")

fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
ax[0].hist(soc, bins=30, color="#4B3F8F"); ax[0].set(title="raw: right-skewed", xlabel="SOC (g/kg)")
ax[1].hist(np.log(soc), bins=30, color="#4B3F8F"); ax[1].set(title="log: roughly symmetric", xlabel="log SOC")
stats.probplot(np.log(soc), dist="norm", plot=ax[2]); ax[2].set(title="Q–Q plot of log SOC")
plt.tight_layout(); plt.show()

Read the printout. The mean sits well above the median, and exponentiating the mean of
the logs lands next to the *median*, not the mean. If you krige log values and simply
exponentiate the result, your map of "mean carbon" is low by over 10% here. Week 2 and Week 5
come back to this.

### Check yourself

**2a.** From `soc`: the fraction of samples above 30 g/kg (an *empirical exceedance
probability*) and the 90th percentile.

**2b.** Write `normal_score(x)`. Rank the values (use `stats.rankdata`), convert
rank $r$ out of $n$ to a probability $(r - 0.5)/n$, and pass that through the normal
quantile function `stats.norm.ppf`.

In [ ]:
p_above_30 = ...
q90 = ...

def normal_score(x):
    # YOUR CODE HERE
    return ...

<details><summary>Show a model answer</summary>

```python
p_above_30 = np.mean(soc > 30)          # a mean of True/False values is a fraction
q90 = np.quantile(soc, 0.9)

def normal_score(x):
    r = stats.rankdata(x)
    return stats.norm.ppf((r - 0.5) / len(x))

print(p_above_30, q90)
```

</details>

In [ ]:
check("2a fraction above 30", p_above_30,
      lambda v: np.isclose(v, (soc > 30).mean()),
      "Compare soc > 30 to get True/False, then take the mean.")
check("2a 90th percentile", q90,
      lambda v: np.isclose(v, np.quantile(soc, 0.9)),
      "np.quantile(data, 0.9).")
ns = normal_score(soc)
check("2b normal scores", ns,
      lambda v: (abs(np.mean(v)) < 0.01 and abs(np.std(v) - 1) < 0.05
                 and stats.spearmanr(v, soc)[0] > 0.999 and abs(stats.skew(v)) < 0.05),
      "Scores should have mean ~0, sd ~1, zero skew, and the same ordering as soc.")

> **Where this goes:** Week 2 *Distributions and transforms*; Week 8 (SGS works in
> normal-score space and back-transforms every realisation); Week 9 (indicators are
> thresholds on the CDF).

---
## 3 · Covariance, correlation, and the identity behind the variogram

### The idea

**Covariance** measures whether two quantities move together:

$$\operatorname{Cov}(X,Y) = E\big[(X-\mu_X)(Y-\mu_Y)\big].$$

It is positive when $X$ tends to be above its mean whenever $Y$ is above its mean.
Divide by both standard deviations and you get **correlation**, which lies between −1
and +1.

Now apply this to a field. Let $X$ be the value at some location and $Y$ the value
a distance $h$ away. For small $h$ they are strongly correlated. For large $h$ they are not.
So correlation becomes a *function of distance*.

The single most useful line of algebra for this unit is:

$$\boxed{\operatorname{Var}(X - Y) = \operatorname{Var}X + \operatorname{Var}Y - 2\operatorname{Cov}(X,Y)}$$

If both locations have the same mean and the same variance $\sigma^2$ (that is what
*stationarity* buys you), then $E[(X-Y)^2] = \operatorname{Var}(X-Y)$, and dividing by two gives

$$\tfrac12 E\big[(X-Y)^2\big] = \sigma^2 - \operatorname{Cov}(X,Y).$$

The left side is **half the mean squared difference** between values $h$ apart. You can
compute it from data without knowing the mean. It is called the **semivariance**, and the
identity says it equals *total variance minus covariance*. When nearby values are
alike, the covariance is high and the semivariance is small. Far apart, the covariance
falls to zero and the semivariance rises to $\sigma^2$, the **sill**. That is the whole
shape of a variogram, derived in two lines.

### Demonstration: correlation as a function of distance

In [ ]:
z = smooth_transect(5000, corr_len=10, rng=np.random.default_rng(30))

fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
for a, h in zip(ax, (2, 15, 60)):
    r = np.corrcoef(z[:-h], z[h:])[0, 1]
    a.scatter(z[:-h][::5], z[h:][::5], s=4, alpha=0.5, color="#4B3F8F")
    a.set(title=f"h = {h} m:  r = {r:.2f}", xlabel="z(x)", ylabel="z(x + h)")
plt.tight_layout(); plt.show()

Each panel pairs every value with the value $h$ metres further along. At 2 m the pairs
lie on a line. At 60 m they are a shapeless cloud. The reader calls these
**h-scatterplots**, and in Week 3 you will turn the cloud's spread into a number.

### Check yourself

**3a.** Verify the boxed identity numerically on the pairs below. Use `np.var` and
`np.cov(..., ddof=0)` so both sides use the same divisor. Store `(lhs, rhs)`.

**3b.** On the long transect `z` above, find the **smallest lag $h$ (in whole metres)
at which the correlation drops below 0.05**. This is roughly what the reader will
call the *practical range*.

In [ ]:
rng = np.random.default_rng(31)
X = rng.normal(10, 2, 500)
Y = 0.7 * X + rng.normal(3, 1.4, 500)

lhs = ...      # Var(X - Y)
rhs = ...      # Var X + Var Y - 2 Cov(X, Y)
identity = (lhs, rhs) if lhs is not ... else ...

h_decorr = ... # smallest h with corr(z[:-h], z[h:]) < 0.05

<details><summary>Show a model answer</summary>

```python
rng = np.random.default_rng(31)
X = rng.normal(10, 2, 500)
Y = 0.7 * X + rng.normal(3, 1.4, 500)

lhs = np.var(X - Y)
rhs = np.var(X) + np.var(Y) - 2 * np.cov(X, Y, ddof=0)[0, 1]
identity = (lhs, rhs)

h_decorr = next(h for h in range(1, 200)
                if np.corrcoef(z[:-h], z[h:])[0, 1] < 0.05)
print(identity, h_decorr)
```

</details>

In [ ]:
check("3a the variance identity", identity,
      lambda v: np.isclose(v[0], v[1]) and np.isclose(v[0], np.var(X - Y)),
      "lhs = np.var(X - Y); for the covariance use np.cov(X, Y, ddof=0)[0, 1].")
check("3b decorrelation distance", h_decorr,
      lambda v: 25 <= v <= 45,
      "Loop h = 1, 2, 3, ... and stop at the first h where the correlation < 0.05.")

For this kernel, theory puts the answer at $2L\sqrt{\ln 20} \approx 35$ m when
$L = 10$ m. One realisation gives something nearby but not exactly that, which is
§1's point again.

> **Where this goes:** Week 3 *From "nearby things are similar" to a number*; Week 4
> (the variogram models are built from covariances); Week 5 (the kriging system is
> written in covariances).

---
## 4 · Distances, and why coordinates must be in metres

### The idea

Everything in geostatistics is a function of the **distance** between locations. That
makes two skills essential.

**Pairwise distances, fast.** With $n$ points there are $n(n-1)/2$ distinct pairs. For
300 samples that is almost 45 000. You build the full $n \times n$ distance matrix with
NumPy **broadcasting**, never with a double loop (task 12 of the reader's Python
self-check).

**Distances in metres, not degrees.** GPS gives longitude and latitude in *degrees*.
One degree of latitude is about 110.6 km. One degree of longitude is about
$111.3\cos(\text{latitude})$ km, which is almost the same near the equator but shrinks
towards the poles. A Euclidean "distance" computed on degrees is therefore in a unit
that has no fixed length. A variogram range of "0.05" means nothing to a reader, and
an anisotropy analysis done in degrees measures the grid, not the soil.

Project first. Most of Kenya uses **UTM zone 37**: **EPSG:32737** (37S) or
**EPSG:32637** (37N). Kenya straddles the equator, so choose *one* zone for the whole
study area and keep to it. Never mix northings from the two hemispheres.

### Demonstration

In [ ]:
import geopandas as gpd
from pyproj import Geod

rng = np.random.default_rng(40)
lon = rng.uniform(36.70, 37.00, 60)
lat = rng.uniform(-1.20, -0.95, 60)
# ten 'duplicate-check' samples a few hundred metres from existing ones
pick = rng.choice(60, 10, replace=False)
lon = np.r_[lon, lon[pick] + rng.uniform(-0.003, 0.003, 10)]
lat = np.r_[lat, lat[pick] + rng.uniform(-0.003, 0.003, 10)]

pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(lon, lat), crs="EPSG:4326")
utm = pts.to_crs("EPSG:32737")
xy = np.column_stack([utm.geometry.x, utm.geometry.y])

d_deg = np.hypot(lon[0] - lon[1], lat[0] - lat[1])
d_m = np.hypot(*(xy[0] - xy[1]))
d_geod = Geod(ellps="WGS84").inv(lon[0], lat[0], lon[1], lat[1])[2]
print(f"'distance' in degrees : {d_deg:.4f}   <- meaningless as a length")
print(f"UTM 37S distance      : {d_m:,.0f} m")
print(f"true geodesic distance: {d_geod:,.0f} m   (UTM error {100*(d_m/d_geod-1):+.3f}%)")

Within a county, the projected distance and the true distance on the ellipsoid agree
to a fraction of a percent. That is why we project once, then treat the plane as
flat.

### Check yourself

**4a.** Build the full distance matrix `D` (shape $70 \times 70$, in metres) from `xy`
using broadcasting. No loops, and no `cdist`: the check uses `cdist` to mark you.

**4b.** Count the **distinct pairs** of samples less than 500 m apart. Each pair counts
once, and a point is not paired with itself.

In [ ]:
D = ...
n_close = ...

<details><summary>Show a model answer</summary>

```python
D = np.linalg.norm(xy[:, None, :] - xy[None, :, :], axis=-1)
# upper triangle (k=1) excludes the diagonal and counts each pair once
n_close = int(np.sum(np.triu(D < 500, k=1)))
print(D.shape, n_close)
```

</details>

In [ ]:
check("4a distance matrix", D,
      lambda v: v.shape == (70, 70) and np.allclose(v, cdist(xy, xy)),
      "xy[:, None, :] - xy[None, :, :] has shape (70, 70, 2); take the norm over the last axis.")
check("4b close pairs", n_close,
      lambda v: v == int(np.sum(np.triu(cdist(xy, xy) < 500, k=1))),
      "Use np.triu(D < 500, k=1) so the diagonal and the duplicate lower half are dropped.")

> **Where this goes:** Week 3 *Binning: the three choices you must make* (you choose
> lag widths *in metres*); Week 4 *Anisotropy*; Week 5 (the kriging matrix is built
> from exactly this `D`).

---
## 5 · Fitting curves: least squares, weights and starting values

### The idea

**Least squares** picks the parameters that make the **sum of squared residuals**
(observed minus fitted) as small as possible.

* When the model is **linear in its parameters**, e.g. a planar trend
  $z = a + b\,x + c\,y$, there is an exact answer: `np.linalg.lstsq`.
* When it is **not**, e.g. a curve that rises and levels off, the answer is found by
  iterative search: `scipy.optimize.curve_fit`. Search needs a **starting guess**
  (`p0`). A bad guess can converge to nonsense or not at all. Sensible **bounds** help.
* **Weighted** least squares lets some points count more than others. In
  `curve_fit` you pass `sigma`, the expected error of each point. Points averaged
  from more raw observations are more reliable, so they get a smaller `sigma`.

A saturating curve fitted by weighted least squares is exactly what you will do with a
variogram in Week 4. Here we practise on a fertiliser trial so the shape is familiar
before it means anything spatial.

### Demonstration: removing a planar trend

In [ ]:
rng = np.random.default_rng(50)
x, y = rng.uniform(0, 20, 150), rng.uniform(0, 20, 150)          # km
zt = 12 + 0.8 * x - 0.3 * y + rng.normal(0, 1.5, 150)            # e.g. mean annual rainfall gradient

A = np.column_stack([np.ones_like(x), x, y])                      # design matrix
coef, *_ = np.linalg.lstsq(A, zt, rcond=None)
resid = zt - A @ coef
print("fitted a, b, c =", coef.round(3), "  (truth 12, 0.8, -0.3)")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
s0 = ax[0].scatter(x, y, c=zt, cmap="viridis"); ax[0].set(title="data: strong trend"); plt.colorbar(s0, ax=ax[0])
s1 = ax[1].scatter(x, y, c=resid, cmap="RdBu_r"); ax[1].set(title="residuals after removing the plane"); plt.colorbar(s1, ax=ax[1])
plt.tight_layout(); plt.show()

### Check yourself

**5a.** Recover the trend coefficients yourself on a fresh dataset `(x2, y2, z2)`.
Store them as an array `trend = [a, b, c]`.

**5b.** A maize trial: nine nitrogen levels, with a different number of plots averaged
at each. Fit the saturating (Mitscherlich) curve

$$ y(N) = b + A\,\big(1 - e^{-N/c}\big) $$

with `curve_fit`. Pass `sigma = 1/np.sqrt(n_plots)` so that better-replicated levels
count more, give sensible `p0` and `bounds`, and store the fitted `(b, A, c)` as `mitsch`.

In [ ]:
rng = np.random.default_rng(51)
x2, y2 = rng.uniform(0, 20, 200), rng.uniform(0, 20, 200)
z2 = 5 - 0.4 * x2 + 1.1 * y2 + rng.normal(0, 2, 200)

N = np.array([0, 25, 50, 75, 100, 125, 150, 175, 200], float)      # kg N / ha
n_plots = np.array([12, 4, 10, 3, 8, 2, 6, 3, 10])
true_bAc = (1.5, 5.0, 45.0)
yield_t = true_bAc[0] + true_bAc[1] * (1 - np.exp(-N / true_bAc[2]))
yield_obs = yield_t + rng.normal(0, 0.9, 9) / np.sqrt(n_plots)     # t/ha, level means

def mitscherlich(N, b, A, c):
    return b + A * (1 - np.exp(-N / c))

plt.errorbar(N, yield_obs, yerr=0.9/np.sqrt(n_plots), fmt="o", color="#4B3F8F")
plt.gca().set(xlabel="nitrogen (kg/ha)", ylabel="yield (t/ha)", title="Level means ± 1 s.e."); plt.show()

In [ ]:
trend = ...
mitsch = ...

<details><summary>Show a model answer</summary>

```python
A2 = np.column_stack([np.ones_like(x2), x2, y2])
trend, *_ = np.linalg.lstsq(A2, z2, rcond=None)

# Starting values read off the plot: baseline ~1.5, gain ~5, levels off by ~100 so c ~ 40
mitsch, _ = optimize.curve_fit(mitscherlich, N, yield_obs, p0=[1.5, 5, 40],
                               sigma=1/np.sqrt(n_plots),
                               bounds=([0, 0, 1], [10, 20, 500]))
print(trend.round(3), mitsch.round(2))
```

</details>

In [ ]:
check("5a trend coefficients", trend,
      lambda v: np.allclose(v, [5, -0.4, 1.1], atol=[1.0, 0.08, 0.08]),
      "Design matrix = columns [1, x2, y2]; np.linalg.lstsq(A, z2, rcond=None)[0].")
check("5b saturating curve", mitsch,
      lambda v: abs(v[0] - 1.5) < 0.6 and abs(v[1] - 5) < 0.8 and abs(v[2] - 45) < 15,
      "curve_fit(mitscherlich, N, yield_obs, p0=[...], sigma=1/np.sqrt(n_plots), bounds=(...)).")

Try 5b again with `p0=[1, 1, 1]` and no bounds, and watch how the fit behaves. Then do
the same with `sigma` removed: the badly replicated levels start pulling the curve.
The variogram version of this lesson in Week 4 costs marks.

> **Where this goes:** Week 2 *Trend, and the decision it forces*; Week 4 *Fitting*
> (weighted least squares on the variogram, with the number of pairs as the weight);
> Week 6 *Regression kriging*.

---
## 6 · Weighted averages, constraints and linear systems

### The idea

Every spatial interpolator you will meet predicts an unknown value as a **weighted
average** of nearby data:

$$\hat z(x_0) = \sum_i w_i\, z(x_i).$$

**Why the weights sum to one.** If every $z(x_i)$ has the same mean $\mu$, then
$E[\hat z] = \mu \sum w_i$. That equals $\mu$, so the estimate is **unbiased**, exactly when
$\sum w_i = 1$.

**Inverse distance weighting (IDW)** sets $w_i \propto 1/d_i^{\,p}$ and rescales them to sum to
one. It is simple, but it looks only at distances to the target. It ignores how the data
relate *to each other*, so two samples on top of each other count twice.

**A better question** is: *which weights, summing to one, make the error variance as
small as possible?* The answer is a small **linear system** to solve with
`np.linalg.solve` (never by inverting the matrix). Below we do it for rain gauges. In
Week 5 exactly the same algebra, with spatial covariances in the matrix, is
**ordinary kriging**.

For errors with covariance matrix $C$, the variance of $\sum w_i e_i$ is $w^\top C\,w$. The
minimising weights with $\sum w_i = 1$ satisfy

$$\begin{bmatrix} C & \mathbf 1 \\ \mathbf 1^\top & 0 \end{bmatrix}
  \begin{bmatrix} w \\ m \end{bmatrix} =
  \begin{bmatrix} \mathbf 0 \\ 1 \end{bmatrix}.$$

The top rows say that at the best weights, moving a little weight from one gauge to
another cannot reduce the variance. $m$ is a **Lagrange multiplier**, a bookkeeping
unknown that the constraint introduces. The bottom row *is* the constraint.

### Demonstration

In [ ]:
def best_weights(C):
    n = len(C)
    K = np.ones((n + 1, n + 1)); K[:n, :n] = C; K[n, n] = 0
    rhs = np.r_[np.zeros(n), 1.0]
    return np.linalg.solve(K, rhs)[:n]

# Two independent gauges; A has error sd 2 mm, B has 4 mm.
C2 = np.diag([2.0**2, 4.0**2])
print("two gauges           :", best_weights(C2).round(3), " (1/sigma^2 rule: 0.8, 0.2)")

# Three equally good gauges (sd 3 mm), but A and B are 100 m apart and share
# the same local storm cell, so their errors are correlated 0.9.
C3 = 9.0 * np.array([[1.0, 0.9, 0.0],
                     [0.9, 1.0, 0.0],
                     [0.0, 0.0, 1.0]])
print("clustered A,B + far C:", best_weights(C3).round(3))

In the second case the three gauges are equally precise, yet C gets about half the
weight, because A and B largely repeat each other. IDW cannot do this. This is
**declustering** coming out of the algebra for free, and it is the main reason kriging
beats IDW on clustered samples.

### A warning: not every matrix can be a covariance matrix

A variance can never be negative, so $w^\top C w \ge 0$ must hold for *every* choice of
weights. Matrices with this property are **positive semi-definite**: all their
eigenvalues are $\ge 0$. Suppose someone draws a correlation-versus-distance curve by hand
that drops off a cliff:

In [ ]:
# 3 points at 0, 1, 2 km; 'correlation' 1 at 0 km, 0.9 at 1 km, 0.2 at 2 km
C_bad = np.array([[1.0, 0.9, 0.2],
                  [0.9, 1.0, 0.9],
                  [0.2, 0.9, 1.0]])
w = np.array([1.0, -2.0, 1.0])
print("eigenvalues:", np.linalg.eigvalsh(C_bad).round(3))
print("variance of z0 - 2 z1 + z2 =", w @ C_bad @ w, "  <- impossible")

A negative variance is not a rounding error. It means the curve cannot describe any
real field. This is why Week 4 restricts you to a short list of **permissible**
variogram models, and why you may not simply join the dots of an empirical variogram.

### Check yourself

**6a.** Write `idw_weights(d, p=2)`, which returns weights proportional to $1/d^p$ and
summing to one.

**6b.** Four gauges with the error covariance matrix `C4` below. Compute the
minimum-variance weights *yourself* (build the bordered matrix and use
`np.linalg.solve`; don't call `best_weights`). Store them as `w4`.

In [ ]:
C4 = np.array([[4.0, 3.0, 0.5, 0.0],
               [3.0, 4.0, 0.5, 0.0],
               [0.5, 0.5, 2.0, 0.2],
               [0.0, 0.0, 0.2, 6.0]])

In [ ]:
def idw_weights(d, p=2):
    # YOUR CODE HERE
    return ...

w4 = ...

<details><summary>Show a model answer</summary>

```python
def idw_weights(d, p=2):
    w = 1.0 / np.asarray(d, float) ** p
    return w / w.sum()

K = np.ones((5, 5)); K[:4, :4] = C4; K[4, 4] = 0
w4 = np.linalg.solve(K, np.r_[0, 0, 0, 0, 1.0])[:4]
print(idw_weights([1, 2]), w4.round(3))
```

</details>

In [ ]:
check("6a IDW weights", idw_weights(np.array([1.0, 2.0])),
      lambda v: np.allclose(v, [0.8, 0.2]),
      "w = 1/d**p, then divide by w.sum(). For d = [1, 2], p = 2 you should get [0.8, 0.2].")
check("6b minimum-variance weights", w4,
      lambda v: np.isclose(np.sum(v), 1) and np.allclose(v, best_weights(C4)),
      "K is 5x5: C4 in the top-left, ones on the border, 0 in the corner; right-hand side [0,0,0,0,1].")

> **Where this goes:** Week 5 *Ordinary kriging* (*What drives the weights* is this
> section with spatial covariances); Week 4 *The permissible models*; Week 2
> *Preferential sampling and declustering*.

---
## 7 · The normal distribution, uncertainty and exceedance

### The idea

Kriging gives two numbers at every location: a **prediction** $\hat z$ and a
**standard deviation** $\sigma$ (the square root of the kriging variance). Under a Gaussian
assumption those two define a full normal distribution for the unknown value, and
three things follow.

* **Standardise:** $z_\text{score} = (\text{value} - \hat z)/\sigma$.
* **Interval:** the true value lies in $\hat z \pm 1.96\,\sigma$ with probability 0.95.
* **Probability of crossing a threshold:**
  $P(Z < t) = \Phi\!\left(\frac{t - \hat z}{\sigma}\right)$, where $\Phi$ is the standard normal CDF
  (`stats.norm.cdf`). Decisions are made on this number ("fertilise where P(SOC deficient) > 0.8"),
  not on the prediction alone.

**Coverage** keeps the method honest. If you claim 95% intervals, about 95% of
held-out true values must fall inside them. Too few means you are **overconfident**,
because your $\sigma$ is too small. Too many means **underconfident**.

### Demonstration

In [ ]:
zhat, sd, t = 20.0, 4.0, 15.0
p = stats.norm.cdf((t - zhat) / sd)
xs = np.linspace(5, 35, 300)
plt.plot(xs, stats.norm.pdf(xs, zhat, sd), color="#4B3F8F")
plt.fill_between(xs[xs < t], stats.norm.pdf(xs[xs < t], zhat, sd), color="#4B3F8F", alpha=0.3)
plt.axvline(t, color="k", lw=0.8)
plt.title(f"prediction 20, sd 4:  P(value < 15) = {p:.3f}"); plt.xlabel("SOC (g/kg)"); plt.show()

### Check yourself

A kriged map of soil carbon has been handed to you as two $60 \times 60$ grids: `mu`
(prediction) and `sdmap` (standard deviation). The standard deviation grows away from
the sample sites.

**7a.** Compute the grid `p_def` of the probability that SOC is **below 15 g/kg**, and
count the cells where that probability exceeds 0.8 (`n_act`).

**7b.** Someone else's model was tested at 400 held-out sites (`truth`, `pred`,
`sd_claimed`). Compute the **coverage** of their 95% intervals, then set `verdict` to
`"overconfident"`, `"underconfident"` or `"about right"`.

In [ ]:
rng = np.random.default_rng(70)
gx, gy = np.meshgrid(np.linspace(0, 1, 60), np.linspace(0, 1, 60))
mu = 18 + 6 * np.sin(3 * gx) * np.cos(2 * gy) - 4 * gy
sdmap = 1.5 + 3.0 * np.hypot(gx - 0.3, gy - 0.4)

truth = rng.normal(20, 5, 400)
true_err_sd = 3.0
pred = truth + rng.normal(0, true_err_sd, 400)
sd_claimed = np.full(400, 1.5)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
plt.colorbar(ax[0].imshow(mu, origin="lower", cmap="viridis"), ax=ax[0]); ax[0].set_title("mu (g/kg)")
plt.colorbar(ax[1].imshow(sdmap, origin="lower", cmap="magma"), ax=ax[1]); ax[1].set_title("sdmap (g/kg)")
plt.tight_layout(); plt.show()

In [ ]:
p_def = ...
n_act = ...

coverage = ...
verdict = ...

<details><summary>Show a model answer</summary>

```python
p_def = stats.norm.cdf((15 - mu) / sdmap)
n_act = int(np.sum(p_def > 0.8))

inside = np.abs(truth - pred) <= 1.96 * sd_claimed
coverage = inside.mean()
verdict = "overconfident"          # far fewer than 95% inside: the claimed sd is too small
print(n_act, round(coverage, 3), verdict)
```

</details>

In [ ]:
check("7a probability grid", p_def,
      lambda v: np.shape(v) == (60, 60) and np.allclose(v, stats.norm.cdf((15 - mu) / sdmap)),
      "stats.norm.cdf((threshold - mu) / sdmap) works element-wise on whole grids.")
check("7a cells to act on", n_act,
      lambda v: v == int(np.sum(stats.norm.cdf((15 - mu) / sdmap) > 0.8)),
      "Count p_def > 0.8 with np.sum.")
check("7b coverage", coverage,
      lambda v: 0.6 < v < 0.75,
      "Fraction of sites where |truth - pred| <= 1.96 * sd_claimed.")
check("7b verdict", verdict,
      lambda v: v == "overconfident",
      "Is the coverage above or below 0.95? Too few inside means the intervals are too narrow.")

> **Where this goes:** Week 5 *The kriging variance, and the trap in it*; Week 9
> *From probability to decision*; Week 8 (simulation answers the same questions without
> assuming each cell is on its own).

---
## 8 · Judging predictions honestly

### The idea

The **error** of a prediction is $e = \hat z - z$. Three summaries of many errors:

| Statistic | Formula | What it tells you |
|---|---|---|
| Mean error (ME) | $\bar e$ | **bias**: are you systematically high or low? Should be ≈ 0 |
| Mean absolute error (MAE) | $\overline{\lvert e\rvert}$ | typical size of an error |
| Root mean square error (RMSE) | $\sqrt{\overline{e^2}}$ | like MAE, but punishes large errors more; always ≥ MAE |

You compute these on data the model **did not see**: a hold-out set, or
**cross-validation**, where each part of the data takes a turn as the test set.

**The spatial trap.** If you pick the test points *at random*, almost every test point
has a close neighbour left in the training set. Predicting it is easy, so the error
looks small. But a map is mostly used to predict *between* samples and *into gaps*.
Holding out whole **blocks** of space asks the honest question.

### Demonstration

In [ ]:
rng = np.random.default_rng(80)
field = smooth_transect(2000, corr_len=25, rng=rng) * 3 + 20      # the true field, 0–2000 m
sx = np.sort(rng.choice(2000, 200, replace=False))                # 200 sample positions
sz = field[sx]

def nn_predict(train_x, train_z, test_x):
    # predict each test point by the nearest training sample
    j = np.abs(test_x[:, None] - train_x[None, :]).argmin(axis=1)
    return train_z[j]

def rmse(e): return np.sqrt(np.mean(e ** 2))

# random 5-fold CV
folds = rng.permutation(200) % 5
err_rand = np.concatenate([nn_predict(sx[folds != k], sz[folds != k], sx[folds == k]) - sz[folds == k]
                           for k in range(5)])
# block 5-fold CV: five contiguous 400 m blocks
blocks = sx // 400
err_block = np.concatenate([nn_predict(sx[blocks != k], sz[blocks != k], sx[blocks == k]) - sz[blocks == k]
                            for k in range(5)])
print(f"random CV RMSE: {rmse(err_rand):.2f}")
print(f"block  CV RMSE: {rmse(err_block):.2f}   <- same model, same data")

Neither number is "the" accuracy. The random-CV figure answers *how well do I predict
close to where I already have data?*, and the block-CV figure answers *how well do I
predict into a 400 m gap?*. Report the one that matches how the map will be used, and
say which it is. Many machine-learning papers report the first and imply the second.

### Check yourself

**8a.** Write `metrics(pred, obs)` returning a dict with keys `"ME"`, `"MAE"`, `"RMSE"`.

**8b.** (Paper, not code.) A colleague reports RMSE = 1.2 from random 10-fold CV on 500
soil samples, and uses it to claim that their map is accurate to ±1.2 g/kg across the
county. The samples are clustered along the roads. Write two sentences on what is wrong,
then open the model answer.

In [ ]:
def metrics(pred, obs):
    # YOUR CODE HERE
    return ...

<details><summary>Show a model answer</summary>

```python
def metrics(pred, obs):
    e = np.asarray(pred) - np.asarray(obs)
    return {"ME": e.mean(), "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean())}

print(metrics([1, 2, 6], [1, 1, 3]))
```

</details>

In [ ]:
m = metrics(np.array([1.0, 2.0, 6.0]), np.array([1.0, 1.0, 3.0]))
check("8a error metrics", m,
      lambda v: np.isclose(v["ME"], 4/3) and np.isclose(v["MAE"], 4/3) and np.isclose(v["RMSE"], np.sqrt(10/3)),
      "Errors are pred - obs = [0, 1, 3]: ME = 4/3, MAE = 4/3, RMSE = sqrt(10/3).")

<details><summary>8b: model answer</summary>

Random CV on road-clustered samples tests almost only locations that have a close
neighbour on the same road, so 1.2 measures accuracy *along roads*. Most of the county
is far from any road sample, where errors will be larger. Only a block or distance-
based hold-out would say how large, so ±1.2 cannot be claimed "across the county".
</details>

> **Where this goes:** Week 7 *Validation, and honesty about error*; Week 10 *Why a random
> forest on coordinates fails*.

---
## Before Week 1: can you say yes to each of these?

- [ ] I can explain why correlated samples carry less information than independent ones, and what an effective sample size is.
- [ ] I know which is larger for skewed data, the mean or the median, and why back-transforming a log mean underestimates.
- [ ] I can write down $\operatorname{Var}(X-Y)$ and get from it to "semivariance = sill − covariance".
- [ ] I never compute distances in degrees, and I know which EPSG code my study area needs.
- [ ] I can fit a curve with `curve_fit`, and I know what `p0`, `bounds` and `sigma` are for.
- [ ] I can explain why interpolation weights sum to one, and solve a small constrained system with `np.linalg.solve`.
- [ ] I can turn a prediction and a standard deviation into an interval and an exceedance probability.
- [ ] I can compute ME, MAE and RMSE, and explain when random cross-validation is misleading.

Any box you cannot tick: go back to that section. If Python got in the way, do the
reader's *Python refresher and self-check* appendix next. See you in Week 1.